# Neural Graphics Ex1: Training Your Own Diffusion Model!

## Setup environment

In [1]:
# We recommend using these utils.
# https://google.github.io/mediapy/mediapy.html
# https://einops.rocks/
!pip install mediapy einops wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.0 MB/s eta 0:00:00


In [2]:
# Import essential modules. Feel free to add whatever you need.
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from tqdm import tqdm

### Seed your work
To be able to reproduce your code, please use a random seed from this point onward.

In [3]:
def seed_everything(seed):
  torch.cuda.manual_seed(seed)
  torch.manual_seed(seed)

YOUR_SEED = 180 # modify if you want
seed_everything(YOUR_SEED)

### Experiment logging (Do Not Modify)

During **training** (Sections 2.3, 3.3, 4.1), metrics and eval images are logged to [Weights & Biases](https://wandb.ai/). Screenshot those for your PDF report.

Post-training experiments (3.4, 4.2) save figures locally — no W&B logging after `finish_run()`.

You do **not** need to implement any W&B code. Batch loss uses a training-step x-axis; epoch loss and in-training eval images use an epoch x-axis.

In [4]:
# Do Not Modify — experiment logging utilities
import wandb
import torchvision

wandb.login()


def start_run(section: str, config: dict):
    run = wandb.init(project="neural-graphics-a2", name=section, config=config)
    wandb.define_metric("global_step", hidden=True)
    wandb.define_metric("epoch", hidden=True)
    wandb.define_metric("train/batch_loss", step_metric="global_step")
    wandb.define_metric("train/epoch_loss", step_metric="epoch")
    wandb.define_metric("eval/*", step_metric="epoch")
    return run


def log_batch_loss(loss: float, global_step: int):
    wandb.log({"global_step": global_step, "train/batch_loss": loss})


def log_epoch_loss(loss: float, epoch: int):
    wandb.log({"epoch": epoch, "train/epoch_loss": loss})


def log_sample_grid(samples: torch.Tensor, key: str, nrow: int = 5, **step_metrics):
    grid = torchvision.utils.make_grid(samples.cpu(), nrow=nrow, normalize=True)
    wandb.log({**step_metrics, key: wandb.Image(grid)})


def log_matplotlib_figure(fig, key: str, **step_metrics):
    wandb.log({**step_metrics, key: wandb.Image(fig)})


def plot_samples_with_labels(
    samples: torch.Tensor,
    labels: torch.Tensor,
    title: str,
    save_path: str | None = None,
    wandb_key: str | None = None,
    nrow: int = 5,
    **step_metrics,
):
    samples = samples.cpu()
    labels = labels.cpu()

    n = samples.shape[0]
    ncol = nrow
    nrows = (n + nrow - 1) // nrow

    fig, axes = plt.subplots(nrows, ncol, figsize=(ncol * 2, nrows * 2))
    axes = axes.flatten()

    for i in range(len(axes)):
        ax = axes[i]
        if i < n:
            ax.imshow(samples[i].squeeze(0), cmap="gray")
            ax.set_title(f"Label: {labels[i].item()}", fontsize=10)
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    if wandb_key is not None:
        log_matplotlib_figure(fig, wandb_key, **step_metrics)
    plt.close()


def finish_run():
    wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eranhermush (eranhermush-tel-aviv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Basic Ops and UNet blocks
**Notations:**  
 * `Conv2D(kernel_size, stride, padding)` is `nn.Conv2d()`  
 * `BN` is `nn.BatchNorm2d()`  
 * `GELU` is `nn.GELU()`  
 * `ConvTranspose2D(kernel_size, stride, padding)` is `nn.ConvTranspose2d()`  
 * `AvgPool(kernel_size)` is `nn.AvgPool2d()`  
 * `Linear` is `nn.Linear()`  
 * `N`, `C`, `W` and `H` are batch size, channels num, weight and height respectively


### Basic Ops

In [5]:
class Conv(nn.Module):
    """
    A convolutional layer that doesn’t change the image
    resolution, only the channel dimension
    Applies nn.Conv2d(3, 1, 1) followed by BN and GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the Conv layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H, W) output tensor.
        """
        return self.net(x)

class DownConv(nn.Module):
    """
        A convolutional layer downsamples the tensor by 2.
        The layer consists of Conv2D(3, 2, 1) followed by BN and GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the DownConv layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H/2, W/2) output tensor.
        """
        return self.net(x)


class UpConv(nn.Module):
    """
    A convolutional layer that upsamples the tensor by 2.
    The layer consists of ConvTranspose2d(4, 2, 1) followed by
    BN and GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the UpConv layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H*2, W*2) output tensor.
        """
        return self.net(x)


class Flatten(nn.Module):
    """
    Average pooling layer that flattens a 7x7 tensor into a 1x1 tensor.
    The layer consists of AvgPool followed by GELU.
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.AvgPool2d(kernel_size=7),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, 7, 7) input tensor.

        Returns:
            (N, C, 1, 1) output tensor.
        """
        return self.net(x)


class Unflatten(nn.Module):
    """
      Convolutional layer that unflattens/upsamples a 1x1 tensor into a
      7x7 tensor. The layer consists of ConvTranspose2D(7, 7, 0)
      followed by BN and GELU.
    """
    def __init__(self, in_channels: int):
        """
        Initializes Unflatten layer
        Args:
            in_channels (int): The number of input channels
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_channels, in_channels, kernel_size=7, stride=7, padding=0),
            nn.BatchNorm2d(in_channels),
            nn.GELU(),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, 1, 1) input tensor.

        Returns:
            (N, in_channels, 7, 7) output tensor.
        """
        return self.net(x)

class FC(nn.Module):
    """
    Fully connected layer, consisting of nn.linear followed by GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the FC layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, out_channels),
            nn.GELU(),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels) input tensor.

        Returns:
            (N, out_channels) output tensor.
        """
        return self.net(x)


### UNet Blocks

In [6]:
class ConvBlock(nn.Module):
    """
    Two consecutive Conv operations.
    Note that it has the same input and output shape as Conv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes ConvBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            Conv(in_channels, out_channels),
            Conv(out_channels, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H, W) output tensor.
        """
        return self.net(x)


class DownBlock(nn.Module):
    """
    DownConv followed by ConvBlock. Note that it has the same input and output
    shape as DownConv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes DownBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            DownConv(in_channels, out_channels),
            ConvBlock(out_channels, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H/2, W/2) output tensor.
        """
        return self.net(x)


class UpBlock(nn.Module):
    """
    UpConv followed by ConvBlock.
    Note that it has the same input and output shape as UpConv
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes UpBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            UpConv(in_channels, out_channels),
            ConvBlock(out_channels, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H*2, W*2) output tensor.
        """
        return self.net(x)

class FCBlock(nn.Module):
    """
    Fully-connected Block, consisting of FC layer followed by Linear layer. Note
    that it has the same input and output shape as FC.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes FCBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        self.net = nn.Sequential(
            FC(in_channels, out_channels),
            nn.Linear(out_channels, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels) input tensor.

        Returns:
            (N, out_channels) output tensor.
        """
        return self.net(x)


## 2. Unconditional Diffusion Framework


### 2.1 UNet architecture

In [7]:
class DenoisingUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_hiddens: int
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_hiddens = num_hiddens
        D = num_hiddens

        # Contracting (down) path
        self.conv_block = ConvBlock(in_channels, D)   # (in, 28, 28) -> (D, 28, 28)
        self.down1 = DownBlock(D, D)                  # (D, 28, 28)  -> (D, 14, 14)
        self.down2 = DownBlock(D, 2 * D)              # (D, 14, 14)  -> (2D, 7, 7)
        self.flatten = Flatten()                      # (2D, 7, 7)   -> (2D, 1, 1)

        # Bottleneck
        self.unflatten = Unflatten(2 * D)             # (2D, 1, 1)   -> (2D, 7, 7)

        # Time-conditioning: embed t (N, 1) into channel-wise biases
        self.fc1_t = FCBlock(1, 2 * D)                # -> t1, added at the (2D, 7, 7) level
        self.fc2_t = FCBlock(1, D)                    # -> t2, added at the (D, 14, 14) level

        # Expanding (up) path. Inputs are doubled by skip-connection concatenation.
        self.up1 = UpBlock(4 * D, D)                  # (4D, 7, 7)   -> (D, 14, 14)
        self.up2 = UpBlock(2 * D, D)                  # (2D, 14, 14) -> (D, 28, 28)
        self.out_conv_block = ConvBlock(2 * D, D)     # (2D, 28, 28) -> (D, 28, 28)
        self.out = nn.Conv2d(D, in_channels, kernel_size=1, stride=1, padding=0)

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            t: (N, 1) normalized time tensor.

        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."
        D = self.num_hiddens

        # Down path (keep intermediates for skip connections).
        x0 = self.conv_block(x)          # (D, 28, 28)
        d1 = self.down1(x0)              # (D, 14, 14)
        d2 = self.down2(d1)             # (2D, 7, 7)

        # Bottleneck.
        latent = self.flatten(d2)        # (2D, 1, 1)
        unflatten = self.unflatten(latent)  # (2D, 7, 7)

        # Time embeddings, reshaped to broadcast over spatial dims.
        t1 = self.fc1_t(t).view(-1, 2 * D, 1, 1)  # (2D, 1, 1)
        t2 = self.fc2_t(t).view(-1, D, 1, 1)      # (D, 1, 1)

        # Inject time at the bottleneck.
        unflatten = unflatten + t1       # (2D, 7, 7)

        # Up path with skip connections (concatenate along channels).
        up1 = self.up1(torch.cat([unflatten, d2], dim=1))  # (4D,7,7) -> (D,14,14)
        up1 = up1 + t2                                     # inject time
        up2 = self.up2(torch.cat([up1, d1], dim=1))        # (2D,14,14) -> (D,28,28)

        out = self.out_conv_block(torch.cat([up2, x0], dim=1))  # (2D,28,28) -> (D,28,28)
        out = self.out(out)                                     # (in,28,28)
        return out


### 2.1 DDPM Forward and Inverse Process


In [8]:
def ddpm_schedule(beta1: float, beta2: float, num_ts: int, device: str = 'cuda') -> dict:
    """Constants for DDPM training and sampling.

    Arguments:
        beta1: float, starting beta value.
        beta2: float, ending beta value.
        num_ts: int, number of timesteps.

    Returns:
        dict with keys:
            betas: linear schedule of betas from beta1 to beta2.
            alphas: 1 - betas.
            alpha_bars: cumulative product of alphas.
    """
    assert beta1 < beta2 < 1.0, "Expect beta1 < beta2 < 1.0."
    betas = torch.linspace(beta1, beta2, num_ts, device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return {
        "betas": betas,
        "alphas": alphas,
        "alpha_bars": alpha_bars,
    }


In [9]:
def ddpm_forward(
    unet: DenoisingUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 1 of the DDPM paper (not including gradient step).

    Args:
        unet: DenoisingUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        num_ts: int, number of timesteps.
    Returns:
        (,) diffusion loss.
    """
    unet.train()
    N = x_0.shape[0]
    device = x_0.device
    alpha_bars = ddpm_schedule["alpha_bars"]

    # t ~ Uniform({1, ..., T}); alpha_bar for timestep t lives at index t-1.
    t = torch.randint(1, num_ts + 1, (N,), device=device)
    ab = alpha_bars[t - 1].view(N, 1, 1, 1)

    # x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps
    eps = torch.randn_like(x_0)
    x_t = ab.sqrt() * x_0 + (1.0 - ab).sqrt() * eps

    # Predict the noise from the noised image and the normalized timestep.
    t_norm = (t.float() / num_ts).view(N, 1)
    eps_hat = unet(x_t, t_norm)

    return nn.functional.mse_loss(eps_hat, eps)


In [10]:
@torch.inference_mode()
def ddpm_sample(
    unet: DenoisingUNet,
    ddpm_schedule: dict,
    img_wh: tuple[int, int],
    batch_size: int,
    num_ts: int
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper.

    Args:
        unet: DenoisingUNet
        ddpm_schedule: dict
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.

    Returns:
        (N, C, H, W) final sample.
    """
    unet.eval()
    betas = ddpm_schedule["betas"]
    alphas = ddpm_schedule["alphas"]
    alpha_bars = ddpm_schedule["alpha_bars"]
    device = betas.device

    w, h = img_wh
    # Start from pure Gaussian noise x_T ~ N(0, I).
    x_t = torch.randn(batch_size, unet.in_channels, h, w, device=device)

    for t in range(num_ts, 0, -1):
        # z ~ N(0, I) for t > 1, else 0.
        z = torch.randn_like(x_t) if t > 1 else torch.zeros_like(x_t)

        # Timestep constants (index t-1 <-> timestep t).
        beta_t = betas[t - 1]
        alpha_t = alphas[t - 1]
        ab_t = alpha_bars[t - 1]
        ab_prev = alpha_bars[t - 2] if t > 1 else torch.ones((), device=device)

        # Predict noise and recover x_0 estimate.
        t_norm = torch.full((batch_size, 1), t / num_ts, device=device)
        eps_hat = unet(x_t, t_norm)
        x0_hat = (x_t - (1.0 - ab_t).sqrt() * eps_hat) / ab_t.sqrt()

        # Posterior mean coefficients + noise term.
        coef_x0 = (ab_prev.sqrt() * beta_t) / (1.0 - ab_t)
        coef_xt = (alpha_t.sqrt() * (1.0 - ab_prev)) / (1.0 - ab_t)
        x_t = coef_x0 * x0_hat + coef_xt * x_t + beta_t.sqrt() * z

    return x_t


In [11]:
# Do Not Modify
class DDPM(nn.Module):
    def __init__(
        self,
        unet: DenoisingUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.betas_range = betas
        self.num_ts = num_ts
        self.p_uncond = p_uncond
        self.ddpm_schedule = ddpm_schedule(betas[0], betas[1], num_ts)

        for k, v in ddpm_schedule(betas[0], betas[1], num_ts).items():
            self.register_buffer(k, v, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        batch_size: int
    ):
        return ddpm_sample(
            self.unet, self.ddpm_schedule, img_wh, batch_size, self.num_ts
        )

### 2.3 Train your denoiser

In [ ]:
# Hyper parameters - Modify if you wish
num_hidden = 128
batch_size = 64
num_epochs = 20
lr = 1e-3
img_wh = (28, 28)
eval_batch_size = 20
T = 300
eval_epochs = [1, 5, 10, 15, 20]

global_step = 0

# Init MNIST data loaders
train_data = MNIST(root='./data', train=True, download=True, transform=ToTensor())
test_data = MNIST(root='./data', train=False, download=True, transform=ToTensor())
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(test_data, batch_size=eval_batch_size, shuffle=True)

# Init denoiser and DDPM wrapper
denosier_unet = DenoisingUNet(in_channels=1, num_hiddens=num_hidden)
ddpm = DDPM(denosier_unet, num_ts=T)

# Optimizer and device setup - Adam optimizer with exponential learning rate decay
optimizer = torch.optim.Adam(ddpm.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(
    optimizer=optimizer, gamma=0.1 ** (1.0 / num_epochs)
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ddpm.to(device)

run = start_run(
    "ddpm",
    config=dict(
        num_hidden=num_hidden,
        batch_size=batch_size,
        num_epochs=num_epochs,
        lr=lr,
        T=T,
        eval_batch_size=eval_batch_size,
    ),
)

for epoch in tqdm(range(num_epochs)):
    ddpm.train()
    epoch_loss = 0.0

    for batch, (data, label) in enumerate(train_loader):
        optimizer.zero_grad()
        data = data.to(device)
        loss = ddpm(data)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        epoch_loss += batch_loss

        log_batch_loss(batch_loss, global_step)
        global_step += 1

        if batch % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}], "
                f"Step [{batch}/{len(train_loader)}], "
                f"Loss: {batch_loss:.4f}"
            )

    avg_epoch_loss = epoch_loss / len(train_loader)
    log_epoch_loss(avg_epoch_loss, epoch + 1)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_epoch_loss:.4f}")
    scheduler.step()

    ddpm.eval()
    if epoch + 1 in eval_epochs:
        # Generate eval_batch_size samples and log the grid to W&B.
        # ddpm.sample is decorated with @torch.inference_mode(); wrap in no_grad
        # as well per the assignment tip to be safe about memory.
        with torch.no_grad():
            samples = ddpm.sample(img_wh, eval_batch_size)
        log_sample_grid(samples, f"eval/samples_epoch_{epoch+1}", epoch=epoch + 1)

finish_run()


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.50MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.23MB/s]


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch [1/20], Step [0/938], Loss: 0.9761
Epoch [1/20], Step [100/938], Loss: 0.0867
Epoch [1/20], Step [200/938], Loss: 0.0772
Epoch [1/20], Step [300/938], Loss: 0.0694
Epoch [1/20], Step [400/938], Loss: 0.0416
Epoch [1/20], Step [500/938], Loss: 0.0494
Epoch [1/20], Step [600/938], Loss: 0.0798
Epoch [1/20], Step [700/938], Loss: 0.0540
Epoch [1/20], Step [800/938], Loss: 0.0404
Epoch [1/20], Step [900/938], Loss: 0.0504
Epoch [1/20] - Average Loss: 0.0615


  5%|▌         | 1/20 [01:54<36:08, 114.11s/it]

Epoch [2/20], Step [0/938], Loss: 0.0500
Epoch [2/20], Step [100/938], Loss: 0.0397
Epoch [2/20], Step [200/938], Loss: 0.0400
Epoch [2/20], Step [300/938], Loss: 0.0384
Epoch [2/20], Step [400/938], Loss: 0.0308
Epoch [2/20], Step [500/938], Loss: 0.0599
Epoch [2/20], Step [600/938], Loss: 0.0546
Epoch [2/20], Step [700/938], Loss: 0.0412
Epoch [2/20], Step [800/938], Loss: 0.0410
Epoch [2/20], Step [900/938], Loss: 0.0387


...

Epoch [18/20] - Average Loss: 0.0266
Epoch [19/20], Step [0/938], Loss: 0.0293
Epoch [19/20], Step [100/938], Loss: 0.0257
Epoch [19/20], Step [200/938], Loss: 0.0249
Epoch [19/20], Step [300/938], Loss: 0.0314
Epoch [19/20], Step [400/938], Loss: 0.0236
Epoch [19/20], Step [500/938], Loss: 0.0338
Epoch [19/20], Step [600/938], Loss: 0.0322
Epoch [19/20], Step [700/938], Loss: 0.0241
Epoch [19/20], Step [800/938], Loss: 0.0272
Epoch [19/20], Step [900/938], Loss: 0.0295


 95%|█████████▌| 19/20 [34:52<01:49, 109.42s/it]

Epoch [19/20] - Average Loss: 0.0264
Epoch [20/20], Step [0/938], Loss: 0.0232
Epoch [20/20], Step [100/938], Loss: 0.0275
Epoch [20/20], Step [200/938], Loss: 0.0234
Epoch [20/20], Step [300/938], Loss: 0.0289
Epoch [20/20], Step [400/938], Loss: 0.0224
Epoch [20/20], Step [500/938], Loss: 0.0282
Epoch [20/20], Step [600/938], Loss: 0.0304
Epoch [20/20], Step [700/938], Loss: 0.0228
Epoch [20/20], Step [800/938], Loss: 0.0252
Epoch [20/20], Step [900/938], Loss: 0.0398
Epoch [20/20] - Average Loss: 0.0266


100%|██████████| 20/20 [36:45<00:00, 110.26s/it]


epoch,▁▁▁▂▂▂▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇███
global_step,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train/batch_loss,▆▆▃█▆▃▂▃▃▂▁▁▅▄▄▃▃▃▄▃▆▃▂▂▃▃▁▃▃▃▁▃▂▃▂▂▃▃▁▃
train/epoch_loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
global_step,18759
train/batch_loss,0.0365
train/epoch_loss,0.02655


## 3. Implementing class-conditioned diffusion framework with CFG


### 3.1 Adding Class-Conditioning to UNet architecture

In [12]:
class ConditionalDenoisingUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        num_hiddens: int,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.num_hiddens = num_hiddens
        D = num_hiddens

        # Contracting (down) path
        self.conv_block = ConvBlock(in_channels, D)   # (in, 28, 28) -> (D, 28, 28)
        self.down1 = DownBlock(D, D)                  # (D, 28, 28)  -> (D, 14, 14)
        self.down2 = DownBlock(D, 2 * D)              # (D, 14, 14)  -> (2D, 7, 7)
        self.flatten = Flatten()                      # (2D, 7, 7)   -> (2D, 1, 1)

        # Bottleneck
        self.unflatten = Unflatten(2 * D)             # (2D, 1, 1)   -> (2D, 7, 7)

        # Time-conditioning: embed t (N, 1) into channel-wise biases.
        self.fc1_t = FCBlock(1, 2 * D)                # -> t1, added at the (2D, 7, 7) level
        self.fc2_t = FCBlock(1, D)                    # -> t2, added at the (D, 14, 14) level

        # Class-conditioning: embed the one-hot class (N, num_classes) into
        # channel-wise scales that modulate the corresponding feature maps.
        self.fc1_c = FCBlock(num_classes, 2 * D)      # -> c1, scales the (2D, 7, 7) level
        self.fc2_c = FCBlock(num_classes, D)          # -> c2, scales the (D, 14, 14) level

        # Expanding (up) path. Inputs are doubled by skip-connection concatenation.
        self.up1 = UpBlock(4 * D, D)                  # (4D, 7, 7)   -> (D, 14, 14)
        self.up2 = UpBlock(2 * D, D)                  # (2D, 14, 14) -> (D, 28, 28)
        self.out_conv_block = ConvBlock(2 * D, D)     # (2D, 28, 28) -> (D, 28, 28)
        self.out = nn.Conv2d(D, in_channels, kernel_size=1, stride=1, padding=0)

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
        t: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N, num_classes) float condition tensor.
            t: (N, 1) normalized time tensor.
            mask: (N, 1) mask tensor. If not None, mask out condition when mask == 0.

        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."
        D = self.num_hiddens

        # Drop the label for masked-out samples: replace the one-hot vector with
        # zeros so those samples represent the unconditional case.
        if mask is not None:
            c = c * mask

        # Down path (keep intermediates for skip connections).
        x0 = self.conv_block(x)          # (D, 28, 28)
        d1 = self.down1(x0)              # (D, 14, 14)
        d2 = self.down2(d1)              # (2D, 7, 7)

        # Bottleneck.
        latent = self.flatten(d2)        # (2D, 1, 1)
        unflatten = self.unflatten(latent)  # (2D, 7, 7)

        # Time and class embeddings, reshaped to broadcast over spatial dims.
        t1 = self.fc1_t(t).view(-1, 2 * D, 1, 1)  # (2D, 1, 1)
        t2 = self.fc2_t(t).view(-1, D, 1, 1)      # (D, 1, 1)
        c1 = self.fc1_c(c).view(-1, 2 * D, 1, 1)  # (2D, 1, 1)
        c2 = self.fc2_c(c).view(-1, D, 1, 1)      # (D, 1, 1)

        # Inject class (scale) and time (shift) at the bottleneck.
        unflatten = c1 * unflatten + t1  # (2D, 7, 7)

        # Up path with skip connections (concatenate along channels).
        up1 = self.up1(torch.cat([unflatten, d2], dim=1))  # (4D,7,7) -> (D,14,14)
        up1 = c2 * up1 + t2                                # inject class + time
        up2 = self.up2(torch.cat([up1, d1], dim=1))        # (2D,14,14) -> (D,28,28)

        out = self.out_conv_block(torch.cat([up2, x0], dim=1))  # (2D,28,28) -> (D,28,28)
        out = self.out(out)                                     # (in,28,28)
        return out


### 3.2 DDPM Forward and Inverse Process with CFG

In [13]:
def ddpm_forward(
    unet: ConditionalDenoisingUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    c: torch.Tensor,
    p_uncond: float,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 3 (not including gradient step).

    Args:
        unet: ConditionalDenoisingUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        c: (N,) int64 condition tensor.
        p_uncond: float, probability of unconditioning the condition.
        num_ts: int, number of timesteps.

    Returns:
        (,) diffusion loss.
    """
    unet.train()
    N = x_0.shape[0]
    device = x_0.device
    alpha_bars = ddpm_schedule["alpha_bars"]

    # Make the integer labels into one-hot vectors (N, num_classes).
    c_onehot = nn.functional.one_hot(c, unet.num_classes).float()

    # Drop the label (unconditional) for each sample with probability p_uncond.
    # mask == 1 keeps the label, mask == 0 drops it.
    mask = (torch.rand(N, device=device) > p_uncond).float().view(N, 1)

    # t ~ Uniform({1, ..., T}); alpha_bar for timestep t lives at index t-1.
    t = torch.randint(1, num_ts + 1, (N,), device=device)
    ab = alpha_bars[t - 1].view(N, 1, 1, 1)

    # x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps
    eps = torch.randn_like(x_0)
    x_t = ab.sqrt() * x_0 + (1.0 - ab).sqrt() * eps

    # Predict the noise from the noised image, class condition and timestep.
    t_norm = (t.float() / num_ts).view(N, 1)
    eps_hat = unet(x_t, c_onehot, t_norm, mask)

    return nn.functional.mse_loss(eps_hat, eps)


In [14]:
@torch.inference_mode()
def ddpm_cfg_sample(
    unet: ConditionalDenoisingUNet,
    ddpm_schedule: dict,
    c: torch.Tensor,
    img_wh: tuple[int, int],
    num_ts: int,
    guidance_scale: float = 5.0
) -> torch.Tensor:
    """Algorithm 4.

    Args:
        unet: ConditionalDenoisingUNet
        ddpm_schedule: dict
        c: (N,) int64 condition tensor. Only for class-conditional
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.
        guidance_scale: float, CFG scale.

    Returns:
        (N, C, H, W) final sample.
    """
    unet.eval()
    betas = ddpm_schedule["betas"]
    alphas = ddpm_schedule["alphas"]
    alpha_bars = ddpm_schedule["alpha_bars"]
    device = betas.device

    N = c.shape[0]
    c_onehot = nn.functional.one_hot(c.to(device), unet.num_classes).float()

    # Masks that select the conditional / unconditional forward passes.
    cond_mask = torch.ones(N, 1, device=device)
    uncond_mask = torch.zeros(N, 1, device=device)

    w, h = img_wh
    # Start from pure Gaussian noise x_T ~ N(0, I).
    x_t = torch.randn(N, unet.in_channels, h, w, device=device)

    for t in range(num_ts, 0, -1):
        # z ~ N(0, I) for t > 1, else 0.
        z = torch.randn_like(x_t) if t > 1 else torch.zeros_like(x_t)

        # Timestep constants (index t-1 <-> timestep t).
        beta_t = betas[t - 1]
        alpha_t = alphas[t - 1]
        ab_t = alpha_bars[t - 1]
        ab_prev = alpha_bars[t - 2] if t > 1 else torch.ones((), device=device)

        # Classifier-free guidance: combine conditional and unconditional noise.
        t_norm = torch.full((N, 1), t / num_ts, device=device)
        eps_cond = unet(x_t, c_onehot, t_norm, cond_mask)
        eps_uncond = unet(x_t, c_onehot, t_norm, uncond_mask)
        eps_hat = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        # Recover x_0 estimate.
        x0_hat = (x_t - (1.0 - ab_t).sqrt() * eps_hat) / ab_t.sqrt()

        # Posterior mean coefficients + noise term.
        coef_x0 = (ab_prev.sqrt() * beta_t) / (1.0 - ab_t)
        coef_xt = (alpha_t.sqrt() * (1.0 - ab_prev)) / (1.0 - ab_t)
        x_t = coef_x0 * x0_hat + coef_xt * x_t + beta_t.sqrt() * z

    return x_t


In [15]:
# Do Not Modify
class DDPM(nn.Module):
    def __init__(
        self,
        unet: ConditionalDenoisingUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.betas = betas
        self.num_ts = num_ts
        self.p_uncond = p_uncond
        self.ddpm_schedule = ddpm_schedule(betas[0], betas[1], num_ts)

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, c, self.p_uncond, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        c: torch.Tensor,
        img_wh: tuple[int, int],
        guidance_scale: float = 5.0
    ):
        return ddpm_cfg_sample(
            self.unet, self.ddpm_schedule, c, img_wh, self.num_ts, guidance_scale
        )

### 3.3 Train your class-conditioned denoiser

In [17]:
# Class-conditioned DDPM training (Section 3.3) — mirrors the Section 2.3 loop.
num_hidden = 128
batch_size = 64
num_epochs = 20
lr = 1e-3
img_wh = (28, 28)
eval_batch_size = 20
T = 300
eval_epochs = [1, 5, 10, 15, 20]

global_step = 0

# Init MNIST data loaders
train_data = MNIST(root='./data', train=True, download=True, transform=ToTensor())
test_data = MNIST(root='./data', train=False, download=True, transform=ToTensor())
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(test_data, batch_size=eval_batch_size, shuffle=True)

# Init denoiser and DDPM wrapper
denosier_unet = DenoisingUNet(in_channels=1, num_hiddens=num_hidden)
ddpm = DDPM(denosier_unet, num_ts=T)

# Optimizer and device setup - Adam optimizer with exponential learning rate decay
optimizer = torch.optim.Adam(ddpm.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(
    optimizer=optimizer, gamma=0.1 ** (1.0 / num_epochs)
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ddpm.to(device)

num_classes = 10
guidance_scale = 5.0

global_step = 0

# Conditional denoiser + conditional DDPM wrapper (cell above).
cond_unet = ConditionalDenoisingUNet(
    in_channels=1, num_classes=num_classes, num_hiddens=num_hidden
)
cond_ddpm = DDPM(cond_unet, num_ts=T, p_uncond=0.1)

# Optimizer and device setup (same recipe as the unconditional model).
optimizer = torch.optim.Adam(cond_ddpm.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(
    optimizer=optimizer, gamma=0.1 ** (1.0 / num_epochs)
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cond_ddpm.to(device)

run = start_run(
    "conditional-ddpm",
    config=dict(
        num_hidden=num_hidden,
        batch_size=batch_size,
        num_epochs=num_epochs,
        lr=lr,
        T=T,
        eval_batch_size=eval_batch_size,
        num_classes=num_classes,
        p_uncond=0.1,
        guidance_scale=guidance_scale,
    ),
)

for epoch in tqdm(range(num_epochs)):
    cond_ddpm.train()
    epoch_loss = 0.0

    for batch, (data, label) in enumerate(train_loader):
        optimizer.zero_grad()
        data = data.to(device)
        label = label.to(device)
        loss = cond_ddpm(data, label)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        epoch_loss += batch_loss

        log_batch_loss(batch_loss, global_step)
        global_step += 1

        if batch % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}], "
                f"Step [{batch}/{len(train_loader)}], "
                f"Loss: {batch_loss:.4f}"
            )

    avg_epoch_loss = epoch_loss / len(train_loader)
    log_epoch_loss(avg_epoch_loss, epoch + 1)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_epoch_loss:.4f}")
    scheduler.step()

    cond_ddpm.eval()
    if epoch + 1 in eval_epochs:
        # Draw a class-labels minibatch from the MNIST test set and generate
        # eval_batch_size samples conditioned on those labels (CFG, gamma=5.0).
        _, eval_labels = next(iter(eval_loader))
        eval_labels = eval_labels.to(device)
        with torch.no_grad():
            samples = cond_ddpm.sample(
                eval_labels, img_wh, guidance_scale=guidance_scale
            )
        plot_samples_with_labels(
            samples,
            eval_labels,
            title=f"Samples after {epoch+1} epochs",
            wandb_key=f"eval/samples_epoch_{epoch+1}",
            epoch=epoch + 1,
        )

finish_run()


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.93MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.7MB/s]


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch [1/20], Step [0/938], Loss: 1.2179
Epoch [1/20], Step [100/938], Loss: 0.0877
Epoch [1/20], Step [200/938], Loss: 0.0441
Epoch [1/20], Step [300/938], Loss: 0.0654
Epoch [1/20], Step [400/938], Loss: 0.1014
Epoch [1/20], Step [500/938], Loss: 0.0610
Epoch [1/20], Step [600/938], Loss: 0.0659
Epoch [1/20], Step [700/938], Loss: 0.0438
Epoch [1/20], Step [800/938], Loss: 0.0374
Epoch [1/20], Step [900/938], Loss: 0.0416
Epoch [1/20] - Average Loss: 0.0688


  5%|▌         | 1/20 [02:03<39:04, 123.38s/it]

Epoch [2/20], Step [0/938], Loss: 0.0480
Epoch [2/20], Step [100/938], Loss: 0.0354
Epoch [2/20], Step [200/938], Loss: 0.0376
Epoch [2/20], Step [300/938], Loss: 0.0375
Epoch [2/20], Step [400/938], Loss: 0.0448
Epoch [2/20], Step [500/938], Loss: 0.0362
Epoch [2/20], Step [600/938], Loss: 0.0412
Epoch [2/20], Step [700/938], Loss: 0.0326
Epoch [2/20], Step [800/938], Loss: 0.0323
Epoch [2/20], Step [900/938], Loss: 0.0295


...

Epoch [18/20] - Average Loss: 0.0255
Epoch [19/20], Step [0/938], Loss: 0.0239
Epoch [19/20], Step [100/938], Loss: 0.0231
Epoch [19/20], Step [200/938], Loss: 0.0240
Epoch [19/20], Step [300/938], Loss: 0.0245
Epoch [19/20], Step [400/938], Loss: 0.0238
Epoch [19/20], Step [500/938], Loss: 0.0268
Epoch [19/20], Step [600/938], Loss: 0.0286
Epoch [19/20], Step [700/938], Loss: 0.0200
Epoch [19/20], Step [800/938], Loss: 0.0264
Epoch [19/20], Step [900/938], Loss: 0.0223


 95%|█████████▌| 19/20 [37:57<01:58, 118.83s/it]

Epoch [19/20] - Average Loss: 0.0255
Epoch [20/20], Step [0/938], Loss: 0.0250
Epoch [20/20], Step [100/938], Loss: 0.0315
Epoch [20/20], Step [200/938], Loss: 0.0294
Epoch [20/20], Step [300/938], Loss: 0.0217
Epoch [20/20], Step [400/938], Loss: 0.0234
Epoch [20/20], Step [500/938], Loss: 0.0169
Epoch [20/20], Step [600/938], Loss: 0.0228
Epoch [20/20], Step [700/938], Loss: 0.0246
Epoch [20/20], Step [800/938], Loss: 0.0261
Epoch [20/20], Step [900/938], Loss: 0.0232
Epoch [20/20] - Average Loss: 0.0255


100%|██████████| 20/20 [40:06<00:00, 120.31s/it]


epoch,▁▁▁▂▂▂▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇███
global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇███
train/batch_loss,▅█▃▄▄▃▂▃▄▅▃▃▂▂▄▂▃▂▁▂▂▂▂▃▃▂▃▃▃▁▁▂▃▂▃▃▃▂▂▁
train/epoch_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
global_step,18759
train/batch_loss,0.03664
train/epoch_loss,0.02547


### 3.4 Experiment with different guidance sacles

In [ ]:
# YOUR CODE HERE.
# Experiment with guidance scales [0, 1, 5, 7, 10, 15]:
#   - Generate one sample per digit (0-9) using ddpm_cfg_sample(...)
#   - Save labeled grids with plot_samples_with_labels(..., save_path=f"guidance_scale_{gamma}.png")
#   - No W&B logging needed here

## 4. Flow Matching (Bonus)

This entire section is optional bonus material. If you choose to complete it, you will implement and train a Flow Matching model on MNIST.

Unlike DDPM, which learns to predict the noise added to an image, Flow Matching learns a velocity field that moves samples from a simple noise distribution toward the data distribution.

You will reuse the same `DenoisingUNet` architecture from Section 2. No architectural changes are required.

### 4.1 Flow Matching Training Objective

In [ ]:
def flow_matching_forward(
    unet: DenoisingUNet,
    x_1: torch.Tensor,
) -> torch.Tensor:
    """
    Flow Matching training objective.

    Args:
        unet: DenoisingUNet.
        x_1: (N, C, H, W) clean images from the dataset.

    Returns:
        (,) flow matching loss.
    """
    unet.train()

    # YOUR CODE HERE:

    raise NotImplementedError()

### 4.2 Flow Matching Sampling

In [ ]:
@torch.inference_mode()
def flow_matching_sample(
    unet: DenoisingUNet,
    img_wh: tuple[int, int],
    batch_size: int,
    num_steps: int,
) -> torch.Tensor:
    """
    Flow Matching sampling using Euler integration.

    Args:
        unet: DenoisingUNet.
        img_wh: (H, W) output image size.
        batch_size: number of samples to generate.
        num_steps: number of Euler integration steps.

    Returns:
        (N, C, H, W) generated samples.
    """
    unet.eval()

    # YOUR CODE HERE:

    raise NotImplementedError()

In [ ]:
# Do Not Modify
class FlowMatching(nn.Module):
    def __init__(
        self,
        unet: DenoisingUNet,
        num_steps: int = 50,
    ):
        super().__init__()
        self.unet = unet
        self.num_steps = num_steps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return flow_matching_forward(self.unet, x)

    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        batch_size: int,
    ) -> torch.Tensor:
        return flow_matching_sample(
            self.unet,
            img_wh,
            batch_size,
            self.num_steps,
        )

### 4.3 Train your Flow Matching model

Train the Flow Matching model on MNIST. Reuse the same `DenoisingUNet` and hyperparameters as Section 2.3 (`num_steps=50` for Euler sampling is the main extra setting).

Submit screenshots from your W&B run:

1. Training loss curves (`train/batch_loss` and `train/epoch_loss`).
2. Evaluation sample grids at epochs 1, 5, 10, 15, and 20.

In [ ]:
# YOUR CODE HERE.
# Train a Flow Matching model (mirror Section 2.3):
#   - Use the same hyperparameters as Section 2.3; set num_steps=50 for Euler sampling
#   - eval at epochs [1, 5, 10, 15, 20]
#   - for W&B logging, use start_run("flow-matching", ...) / log_batch_loss / log_sample_grid / finish_run()